# Partie 2 - Prédiction du nombre d'exacerbations

## 1. Importer les données

In [1]:
import polars as pl

In [2]:
data_path = './data/trainDataDC2.csv.gz'

In [3]:
df = pl.read_csv(data_path)
df.shape

(14575, 21)

In [4]:
df.head()

patid,index_age,previous_asthma_drugs,total_pre_index_cannisters_365,post_index_exacerbations365,pneumonia,sinusitis,acute_bronchitis,acute_laryngitis,upper_respiratory_infection,gerd,rhinitis,adherence,total_pre_index_charge,pre_asthma_days,pre_asthma_charge,pre_asthma_pharma_charge,drug_s,female,log_charges,log_asthma_charge
i64,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
1073754155,"""14""","""1""","""1""",0,"""0""","""0""","""1""","""0""","""1""","""0""","""0""","""NA""","""1224.7674731""","""1""","""314""","""218.13""","""0""","""NA""","""7.1105062871""","""5.3850912153"""
1073799394,"""NA""","""1""","""2""",2,"""0""","""1""","""1""","""0""","""0""","""0""","""1""","""NA""","""20290.534269""","""0""","""NA""","""44.98""","""0""","""0""","""9.9179097641""","""3.8062179465"""
1073854918,"""62""","""1""","""NA""",0,"""0""","""0""","""NA""","""0""","""0""","""0""","""0""","""0.7384196185""","""2964.2541751""","""NA""","""480""","""99.26""","""0""","""0""","""7.9943807368""","""4.5977426702"""
1073898249,"""NA""","""1""","""2""",0,"""0""","""0""","""1""","""0""","""1""","""NA""","""0""","""0.0844686649""","""3223.7088196""","""0""","""0""","""59.58""","""0""","""1""","""8.0782877831""","""NA"""
1073913003,"""40""","""1""","""1""",0,"""0""","""1""","""0""","""1""","""1""","""0""","""1""","""0.5068119891""","""1287.2543683""","""2""","""689""","""NA""","""0""","""0""","""7.1602668324""","""3.3941727667"""


In [5]:
df['patid'].n_unique()

14575

Notre objectif est de construire un *pipeline* de *machine learning* qui pourra fonctionner sur un fichier de même structure que ce fichier de données initial ("trainDataDC2.csv.gz") et qui donnera, en sortie, la prédiction du nombre d'exacerbations de l'asthme, pour chaque patient, dans l'année suivant le début du traitement (variable `post_index_exacerbations365`).

In [6]:
df['post_index_exacerbations365'].null_count()

0

In [7]:
print(
    "Fraction of patients with zero exacerbation: {0:.2%}".format(
        (df['post_index_exacerbations365'] == 0).sum() / df['post_index_exacerbations365'].len()
    )
)

Fraction of patients with zero exacerbation: 88.60%


La variable dépendante (que l'on veut prédire, *outcome*, *label*, *target*, notée $y$) est donc `post_index_exacerbations365`, qui prend des valeurs entières positives.

In [8]:
target = df['post_index_exacerbations365']

Dans une perspective de modélisation statistique, on pense à la **loi de Poisson**
(qui décrit le résultat d'un processus de comptage) :
$$
y \sim \mathrm{Poisson}(\lambda)
$$

* Paramètre $\lambda \in \;]0, +\infty[$
* Espérance $\lambda$
* Variance $\lambda$

$$
\ln(\lambda) = \beta_0 + \beta_1 x
$$

À cause du logarithme, on n'a pas une régression linéaire ordinaire ; on parle de
*generalized linear model (GLM)*.

Il s'agit alors d'estimer :

* $\beta_1$ : coefficient de régression
* $\beta_0$ : intercept (ordonnée à l'origine)

$$
\lambda = \mathrm{e}^{\beta_0 + \beta_1 x} = \mathrm{e}^{\beta_0} \; \mathrm{e}^{\beta_1 x}
$$

## 2. Partitionner les données

Nous commençons par partitionner (*split*) les données, afin d'entraîner (*train*) le modèle sur une partie des données, et de garder une autre partie des données pour tester (*test*) le modèle.

In [9]:
import sklearn as skl

### Séparer les *features* ($X$) et l'*outcome* ($y$)

In [10]:
df.columns

['patid',
 'index_age',
 'previous_asthma_drugs',
 'total_pre_index_cannisters_365',
 'post_index_exacerbations365',
 'pneumonia',
 'sinusitis',
 'acute_bronchitis',
 'acute_laryngitis',
 'upper_respiratory_infection',
 'gerd',
 'rhinitis',
 'adherence',
 'total_pre_index_charge',
 'pre_asthma_days',
 'pre_asthma_charge',
 'pre_asthma_pharma_charge',
 'drug_s',
 'female',
 'log_charges',
 'log_asthma_charge']

Comme dans la partie 1, nous considérons toutes les autres variables comme les caractéristiques (variables indépendantes, *features*, notées $X$), à l'exception de l'identifiant patient `patid` et les variables transformées `log_*`.

In [11]:
features = df.select([
 'index_age',
 'previous_asthma_drugs',
 'total_pre_index_cannisters_365',
 'pneumonia',
 'sinusitis',
 'acute_bronchitis',
 'acute_laryngitis',
 'upper_respiratory_infection',
 'gerd',
 'rhinitis',
 'adherence',
 'total_pre_index_charge',
 'pre_asthma_days',
 'pre_asthma_charge',
 'pre_asthma_pharma_charge',
 'drug_s',
 'female',
])

In [12]:
X_train, X_test, y_train, y_test = skl.model_selection.train_test_split(
    features,
    target,
    test_size=0.2,
    random_state=42,
)

On pourrait vouloir stratifier sur $y$ (`stratify=target`), comme il aurait été souhaitable de le faire dans le cas binaire, sauf qu'ici il y a des classes ($y = 12$ ou $y = 13$) qui ne contiennent qu'un seul membre...

In [13]:
import numpy as np

In [14]:
target.hist(bins=np.arange(target.max() + 1))

breakpoint,category,count
f64,cat,u32
1.0,"""[0.0, 1.0]""",14118
2.0,"""(1.0, 2.0]""",265
3.0,"""(2.0, 3.0]""",95
4.0,"""(3.0, 4.0]""",39
5.0,"""(4.0, 5.0]""",24
…,…,…
9.0,"""(8.0, 9.0]""",3
10.0,"""(9.0, 10.0]""",2
11.0,"""(10.0, 11.0]""",0


## 3. Préparer les données

### Extraire les *features*

Comme nous l'avons déjà vu, notre jeu de données contient des *features* de types hétérogènes. Nous voulons normaliser (centrer et adimensionner) les variables numériques et encoder *one-hot* les variables catégorielles.

In [15]:
X_train.head()

index_age,previous_asthma_drugs,total_pre_index_cannisters_365,pneumonia,sinusitis,acute_bronchitis,acute_laryngitis,upper_respiratory_infection,gerd,rhinitis,adherence,total_pre_index_charge,pre_asthma_days,pre_asthma_charge,pre_asthma_pharma_charge,drug_s,female
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""18""","""1""","""NA""","""0""","""0""","""NA""","""0""","""0""","""0""","""1""","""0.4959128065""","""15132.127497""","""2""","""883""","""1007.78""","""1""","""1"""
"""21""","""1""","""1""","""0""","""1""","""0""","""0""","""1""","""0""","""1""","""0.0844686649""","""2674.9944611""","""3""","""362""","""21.41""","""0""","""0"""
"""53""","""1""","""0""","""1""","""0""","""0""","""0""","""0""","""0""","""0""","""0.6920980926""","""1907.8437372""","""0""","""0""","""225.82""","""0""","""1"""
"""31""","""1""","""NA""","""0""","""NA""","""0""","""0""","""0""","""1""","""1""","""0.1689373297""","""1764.4920051""","""0""","""0""","""21.41""","""0""","""0"""
"""55""","""NA""","""1""","""NA""","""0""","""1""","""0""","""NA""","""NA""","""1""","""0.0844686649""","""1892.0951822""","""0""","""0""","""21.41""","""0""","""0"""


In [16]:
categorical_features = [
 'previous_asthma_drugs',
 'pneumonia',
 'sinusitis',
 'acute_bronchitis',
 'acute_laryngitis',
 'upper_respiratory_infection',
 'gerd',
 'rhinitis',
 'drug_s',
 'female',
]

In [17]:
encoder = skl.preprocessing.OneHotEncoder()

In [18]:
encoder.fit(X_train.select(categorical_features))  # fit encoder to categorical features

,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a :class:`scipy.sparse.csr_matrix`,i.e. a sparse matrix in ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",True
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide `.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'error'
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide `.",None
,"max_catego

In [19]:
encoder.categories_

[array(['1', 'NA'], dtype=object),
 array(['0', '1', 'NA'], dtype=object),
 array(['0', '1', 'NA'], dtype=object),
 array(['0', '1', 'NA'], dtype=object),
 array(['0', '1', 'NA'], dtype=object),
 array(['0', '1', 'NA'], dtype=object),
 array(['0', '1', 'NA'], dtype=object),
 array(['0', '1', 'NA'], dtype=object),
 array(['0', '1', 'NA'], dtype=object),
 array(['0', '1', 'NA'], dtype=object)]

In [20]:
encoder.feature_names_in_

array(['previous_asthma_drugs', 'pneumonia', 'sinusitis',
       'acute_bronchitis', 'acute_laryngitis',
       'upper_respiratory_infection', 'gerd', 'rhinitis', 'drug_s',
       'female'], dtype=object)

Passons aux variables numériques (quantitatives).

In [21]:
numeric_features_int = [
 'index_age',
 'total_pre_index_cannisters_365',
 'pre_asthma_days',
 'pre_asthma_charge',
]

In [22]:
numeric_features_float = [
 'adherence',
 'total_pre_index_charge',
 'pre_asthma_pharma_charge',
]

In [23]:
assert (
    len(categorical_features) +
    len(numeric_features_int) +
    len(numeric_features_float) == len(features.columns)
)

In [24]:
scaler = skl.preprocessing.StandardScaler()

In [25]:
X_train_float = X_train.select(
    numeric_features_float
).cast(
    pl.Float64, strict=False
)

In [26]:
scaler.fit(X_train_float)  # compute mean and std to be used for later scaling

,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [27]:
scaler.feature_names_in_

array(['adherence', 'total_pre_index_charge', 'pre_asthma_pharma_charge'],
      dtype=object)

In [28]:
float_transformer = skl.pipeline.Pipeline(  # sequence of data transformers
    steps=[
        # add conversion to float64,
        ('scaler', skl.preprocessing.StandardScaler()),
        # another preprocessing step
    ]
)

int_transformer = skl.pipeline.Pipeline(
    steps=[
        ('scaler', skl.preprocessing.StandardScaler()),
    ]
)

categorical_transformer = skl.pipeline.Pipeline(
    steps=[
        ('encoder', skl.preprocessing.OneHotEncoder()),
    ]
)

In [29]:
def cast_numeric_int(s):  # user-defined function
    return s.cast(pl.Int64, strict=False)

In [30]:
# construct a data transformer from a user-defined function
cast_int_transformer = skl.preprocessing.FunctionTransformer(cast_numeric_int)

In [31]:
int_transformer = skl.pipeline.Pipeline(
    steps=[
        ('caster', cast_int_transformer),
        ('scaler', skl.preprocessing.StandardScaler()),
    ]
)

In [32]:
def cast_numeric_float(s):
    return s.cast(pl.Float64, strict=False)

cast_float_transformer = skl.preprocessing.FunctionTransformer(cast_numeric_float)

float_transformer = skl.pipeline.Pipeline(
    steps=[
        ('caster', cast_float_transformer),
        ('scaler', skl.preprocessing.StandardScaler()),
    ]
)

In [33]:
preprocessor = skl.compose.ColumnTransformer(  # apply transformers to columns
    transformers=[
        ('num_float', float_transformer, numeric_features_float),
        ('num_int', int_transformer, numeric_features_int),
        ('cat', categorical_transformer, categorical_features),
    ]
)

In [34]:
preprocessor.fit(X_train)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_float', ...), ('num_int', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature

Cet estimateur permet de transformer séparément différentes colonnes des données d'entrée. Les *features* ainsi générées sont alors concaténées pour former un seul *feature space*.

In [35]:
preprocessor.fit_transform(X_train)

array([[ 1.05303977,  0.29940817,  1.71832217, ...,  0.        ,
         1.        ,  0.        ],
       [-0.73360197, -0.26252957, -0.49997531, ...,  1.        ,
         0.        ,  0.        ],
       [ 1.90494841, -0.29713552, -0.0402673 , ...,  0.        ,
         1.        ,  0.        ],
       ...,
       [-0.36680797,  0.09351425, -0.32876269, ...,  0.        ,
         1.        ,  0.        ],
       [ 1.05303977, -0.33332881,  0.14072825, ...,  0.        ,
         1.        ,  0.        ],
       [-0.36680797, -0.36040896, -0.49997531, ...,  0.        ,
         0.        ,  1.        ]], shape=(11660, 36))

La variable d'*outcome* doit-elle être normalisée ?

In [36]:
y_train.shape

(11660,)

In [37]:
y_train.mean()

0.16818181818181818

In [38]:
y_train.median()

0.0

In [39]:
y_train.sum()

1961

## 4. Construire un modèle de prédiction

Ajoutons un modèle prédictif à notre *pipeline*.

In [40]:
pr = skl.linear_model.PoissonRegressor()

In [41]:
X_toy = np.array([[0, 0, 0],
                  [0, 0, 1],
                  [0, 1, 1],
                  [1, 1, 1]])
y_toy = np.array([1, 2, 3, 4])

In [42]:
pr.fit(X_toy, y_toy)

,"alpha alpha: float, default=1Constant that multiplies the L2 penalty term and determines theregularization strength. ``alpha = 0`` is equivalent to unpenalizedGLMs. In this case, the design matrix `X` must have full column rank(no collinearities).Values of `alpha` must be in the range `[0.0, inf)`.",1.0
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the linear predictor (`X @ coef + intercept`).",True
,"solver solver: {'lbfgs', 'newton-cholesky'}, default='lbfgs'Algorithm to use in the optimization problem:'lbfgs' Calls scipy's L-BFGS-B optimizer.'newton-cholesky' Uses Newton-Raphson steps (in arbitrary precision arithmetic equivalent to iterated reweighted least squares) with an inner Cholesky based solver. This solver is a good choice for `n_samples` >> `n_features`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features` because it explicitly computes the Hessian matrix. .. versionadded:: 1.2",'lbfgs'
,"max_iter max_iter: int, default=100The maximal number of iterations for the solver.Values must be in the range `[1, inf)`.",100
,"tol tol: float, default=1e-4Stopping criterion. For the lbfgs solver,the iteration will stop when ``max{|g_j|, j = 1, ..., d} <= tol``where ``g_j`` is the j-th component of the gradient (derivative) ofthe objective function.Values must be in the range `(0.0, inf)`.",0.0001
,"warm_start warm_start: bool, default=FalseIf set to ``True``, reuse the solution of the previous call to ``fit``as initialization for ``coef_`` and ``intercept_`` .",False
,"verbose verbose: int, default=0For the lbfgs solver set verbose to any positive number for verbosity.Values must be in the range `[0, inf)`.",0


In [43]:
pr.score(X_toy, y_toy)  # accuracy

np.float64(0.7236112218520164)

In [44]:
pr.predict(np.array([[0, 0, 0], [0, 0, 1]]))

array([1.78408827, 2.17042105])

In [45]:
pr.coef_

array([0.17743707, 0.23860143, 0.19601367])

In [46]:
pr.intercept_

np.float64(0.5789075118990583)

In [47]:
pr.predict(np.array([[1, 1, 1], [0, 2, 3]]))

array([3.29023726, 5.17661502])

In [48]:
dummy = skl.pipeline.Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('regressor', skl.dummy.DummyRegressor(strategy='mean')),
    ]
)

In [49]:
dummy.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_float', ...), ('num_int', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the differ

Il faut faire mieux que le modèle de régression *dummy* !

In [50]:
reg = skl.pipeline.Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('regressor', pr),
    ]
)

# reg.fit(X_train, y_train)
# ValueError: Input X contains NaN.
# PoissonRegressor does not accept missing values encoded as NaN natively.
# For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier
# and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible
# to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples
# with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of
# all estimators that handle NaN values at the following page:
# https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

Pour utiliser la régression de Poisson, il nous faut remplacer les valeurs numériques manquantes par des valeurs moyennes, médianes ou constantes, etc. (*impute*).

### Exercice possible

Envisager de filtrer (*filter out*) les patients qui ont ("trop" ?) de valeurs manquantes ?

In [51]:
float_transformer = skl.pipeline.Pipeline(  # sequence of data transformers
    steps=[
        ('caster', cast_float_transformer),
        ('imputer', skl.impute.SimpleImputer()),
        ('scaler', skl.preprocessing.StandardScaler()),
    ]
)

int_transformer = skl.pipeline.Pipeline(
    steps=[
        ('caster', cast_int_transformer),
        ('imputer', skl.impute.SimpleImputer()),
        ('scaler', skl.preprocessing.StandardScaler()),
    ]
)

In [52]:
preprocessor = skl.compose.ColumnTransformer(  # apply transformers to columns
    transformers=[
        ('num_float', float_transformer, numeric_features_float),
        ('num_int', int_transformer, numeric_features_int),
        ('cat', categorical_transformer, categorical_features),
    ]
)

In [53]:
reg = skl.pipeline.Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('regressor', pr),
    ]
)

## 5. Entraîner le modèle

In [54]:
reg.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_float', ...), ('num_int', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the differ

### Exercice possible

Explorer l'utilisation du paramètre `sample_weight`.

Utilisons ce modèle entraîné pour prédire les *outcomes* du jeu de test.

In [55]:
y_pred = reg.predict(X_test)

In [56]:
y_pred.min()

np.float64(0.13943624996146195)

In [57]:
y_pred.max()

np.float64(0.24559451580398856)

In [58]:
y_pred.mean()

np.float64(0.16818469094280933)

In [59]:
y_test.mean()

0.20308747855917667

In [60]:
y_pred.sum()

np.float64(490.2583740982892)

In [61]:
y_test.sum()

592

## 6. Évaluer le modèle

In [62]:
skl.metrics.mean_poisson_deviance(y_test, y_pred)

0.9483154871959726